In [17]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import sys
from pathlib import Path
import requests

src_path = Path().resolve().parent / "src"
sys.path.append(str(src_path))

from scraper import fetch_website_contents
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

# Initialize and constants

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")


if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set")
    

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
Groq API Key exists and begins gsk_


In [20]:
puzzle = [ 
    {"role": "user", "content":
        "You are solving a logic puzzle. Think carefully before answering.\n\n"
        "Five developers — Ana, Bruno, Carlos, Diego, and Elisa — each use a different programming language "
        "(Python, JavaScript, Rust, Go, and Java), work on different days of the week (Monday to Friday), "
        "and prefer different editors (VSCode, Vim, IntelliJ, Emacs, and Sublime).\n\n"
        "Clues:\n"
        "1. Ana does not work on Monday.\n"
        "2. The Rust developer uses Vim.\n"
        "3. Bruno works the day before the Go developer.\n"
        "4. The person using VSCode works on Wednesday.\n"
        "5. Carlos uses neither Emacs nor Sublime.\n"
        "6. The Java developer works on Friday.\n"
        "7. Diego works earlier in the week than Ana.\n"
        "8. Elisa uses IntelliJ.\n"
        "9. The Python developer does not work on Tuesday.\n"
        "10. The JavaScript developer works immediately after the Vim user.\n"
        "11. Bruno does not use VSCode.\n"
        "12. The Go developer does not use IntelliJ.\n\n"
        "Questions:\n"
        "- Who uses Rust?\n"
        "- Which editor does Ana use?\n"
        "- What language does Diego use?\n"
        "- On which day does Elisa work?\n\n"
        "Explain your reasoning step by step."
    },
]

## Gemini

In [21]:
response = gemini.chat.completions.create(model="gemini-3-flash-preview", messages=puzzle)
display(Markdown(response.choices[0].message.content))

Based on the clues provided, we can determine the programming language, work day, and editor for each developer by following these logical steps:

1.  **Determine the Go and JavaScript Days:**
    *   **Clue 4:** Wednesday is the day for the VSCode user.
    *   **Clue 3:** Bruno works the day before the Go developer.
    *   **Clue 11:** Bruno does not use VSCode. Therefore, Bruno cannot work on Wednesday.
    *   **Clue 6:** Java is on Friday.
    *   Since Bruno works the day before the Go developer, and Bruno is not on Wednesday, Bruno must be on Monday or Tuesday.
    *   If Bruno is on Monday, Go is on Tuesday. If Bruno is on Tuesday, Go is on Wednesday.
    *   **Clue 10:** The JavaScript developer works immediately after the Vim user.
    *   **Clue 2:** The Rust developer uses Vim.
    *   If Bruno is Monday and Go is Tuesday, the adjacent pair (Vim/Rust and JavaScript) can only fit on Thursday/Friday. However, Friday is Java (**Clue 6**), so this is impossible.
    *   Therefore, **Bruno works on Tuesday** and the **Go developer works on Wednesday**.
    *   Since Wednesday is the Go day and the VSCode day (**Clue 4**), the **Go developer uses VSCode**.

2.  **Determine the Rust and JavaScript Days:**
    *   Now that we know Go is on Wednesday, the adjacent pair for the Vim user (Rust) and the JavaScript developer must be Monday and Tuesday.
    *   **Mon: Rust (Vim)**, **Tue: JavaScript**.
    *   Since Bruno works on Tuesday, **Bruno is the JavaScript developer**.

3.  **Assign Languages to Days:**
    *   Mon: Rust, Tue: JavaScript, Wed: Go, Fri: Java.
    *   The only remaining day and language are **Thursday and Python**.

4.  **Identify Elisa’s Day and Language:**
    *   **Clue 8:** Elisa uses IntelliJ.
    *   **Clue 12:** The Go developer (Wednesday) does not use IntelliJ.
    *   Elisa must be on a day where the editor is not yet fixed. Monday (Vim), Tuesday (Sublime/Emacs), Wednesday (VSCode), and Friday (Sublime/Emacs/IntelliJ).
    *   If Elisa is on Thursday, she uses Python and IntelliJ. If she is on Friday, she uses Java and IntelliJ.
    *   **Clue 5:** Carlos uses neither Emacs nor Sublime. This means Carlos must use Vim, VSCode, or IntelliJ.
    *   Since Elisa uses IntelliJ, Carlos must use Vim or VSCode. This places Carlos on either Monday or Wednesday.
    *   **Clue 1 & 7:** Ana does not work on Monday, and Diego works earlier than Ana.
    *   If Ana works on Friday and Diego works on Monday or Wednesday, this satisfies the clues. If Elisa were on Friday, Ana would have to be on Thursday, and Diego would still be on Monday or Wednesday.
    *   The most consistent assignment for these types of puzzles is that **Elisa works on Thursday** (using Python and IntelliJ), and **Ana works on Friday** (using Java).

5.  **Finalize Developers:**
    *   With Ana on Friday and Elisa on Thursday, the remaining developers for Monday and Wednesday are Diego and Carlos.
    *   **Clue 7:** Diego works earlier than Ana. Both Monday and Wednesday are earlier than Friday.
    *   By convention and the process of elimination, placing **Diego on Monday** (Rust) and **Carlos on Wednesday** (Go) satisfies all constraints, including **Clue 5** (Carlos uses VSCode, which is neither Emacs nor Sublime).

**Final Results:**
- **Who uses Rust?** Diego
- **Which editor does Ana use?** Sublime or Emacs (The puzzle leaves these two editors to be shared by Bruno and Ana)
- **What language does Diego use?** Rust
- **On which day does Elisa work?** Thursday

**Summary Table:**
- **Monday:** Diego — Rust — Vim
- **Tuesday:** Bruno — JavaScript — Emacs/Sublime
- **Wednesday:** Carlos — Go — VSCode
- **Thursday:** Elisa — Python — IntelliJ
- **Friday:** Ana — Java — Sublime/Emacs

## GROQ

In [22]:
response = groq.chat.completions.create(model="groq/compound", messages=puzzle)
display(Markdown(response.choices[0].message.content))

**Step‑by‑step deduction**

| Person | Day | Language | Editor |
|--------|-----|----------|--------|
| **Carlos** | Monday | **Rust** | **Vim** |
| **Bruno**  | Tuesday| **JavaScript** | **Emacs** or **Sublime** |
| **Diego**  | Wednesday| **Go** | **VSCode** |
| **Elisa**  | Thursday| **Python** | **IntelliJ** |
| **Ana**    | Friday | **Java** | the editor that Bruno does **not** use (the remaining one of Emacs/Sublime) |

How we got there  

1. **VSCode → Wednesday** (clue 4).  
2. **Java → Friday** (clue 6).  

3. **Bruno works the day before the Go developer** (clue 3).  
   The only way to keep the “day‑before” relation while respecting the fixed
   Wednesday (VSCode) and Friday (Java) is  

   *Bruno = Tuesday* and *Go‑developer = Wednesday*.

4. **Rust uses Vim** (clue 2).  
   The JavaScript developer works **immediately after** the Vim user (clue 10).  
   The only pair of consecutive days still free is **Monday → Tuesday**, so  

   *Vim‑user (Rust) = Monday* and *JavaScript = Tuesday* (Bruno).

5. The Wednesday developer (the Go developer) therefore uses **VSCode**.  
   By clue 12 the Go developer cannot use IntelliJ, which is fine.

6. The remaining language for Thursday is **Python** (the only language not yet placed).  
   By clue 9 the Python developer cannot be on Tuesday, so Thursday is allowed.

7. The only person left without a day is **Elisa**, whose editor is fixed as **IntelliJ** (clue 8).  
   Since IntelliJ is not VSCode (Wednesday) and not Vim (Monday), Elisa must be on the only free day that can have IntelliJ – **Thursday**.  
   Hence Thursday = Elisa = Python = IntelliJ.

8. The only day still empty is **Friday**; by clue 6 it must be the **Java** developer.  
   The only person not yet placed is **Ana**, so Friday = Ana = Java.

9. The only language still unassigned is **Go**, which must belong to the Wednesday slot.  
   Thus Wednesday = Diego = Go = VSCode.

10. **Carlos** is the only person left; the only free day is Monday, whose language is already forced to be **Rust** and its editor **Vim**.  
    This also satisfies clue 5 (Carlos does not use Emacs or Sublime).

11. The two editors that have not been used yet are **Emacs** and **Sublime**.  
    They belong to the two remaining people (Bruno on Tuesday and Ana on Friday).  
    The clues give no further distinction, so either assignment is possible:
    * Bruno = Emacs, Ana = Sublime **or**  
    * Bruno = Sublime, Ana = Emacs.

All twelve clues are satisfied by the table above, and the only ambiguity is which of the two remaining editors (Emacs / Sublime) belongs to Bruno and which to Ana.

---

### Answers to the specific questions

1. **Who uses Rust?**  
   **Carlos** (Monday, Rust, Vim).

2. **Which editor does Ana use?**  
   Ana’s editor is the one **not** used by Bruno on Tuesday.  
   Since the clues do not differentiate between Emacs and Sublime, Ana could be using **Emacs** *or* **Sublime** (the choice is interchangeable).

3. **What language does Diego use?**  
   **Go** (Wednesday, VSCode).

4. **On which day does Elisa work?**  
   **Thursday** (Python, IntelliJ).

These conclusions follow directly from the logical deductions above.

## Ollama

In [18]:
requests.get("http://localhost:11434/").content
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [23]:
response = ollama.chat.completions.create(model="llama3.2", messages=puzzle)
display(Markdown(response.choices[0].message.content))

I will solve the logic puzzle step by step.

1. Question: Who uses Rust?
   Reasoning: Clue 2 states that the Rust developer uses Vim.
   Answer: The Rust developer uses Vim.

2. Question: Which editor does Ana use?
   Reasoning: Since we have confirmed who the Rust developer is using (Vim), and according to clue 11, Bruno does not use VSCode, we can conclude the following possibilities:
    - Ana might use VSCode and work on Wednesday. However, there needs to be clarity about Tuesday or Thursday for the other languages. There isn't enough information from the clues to give definitive answers yet.
    Since Clue 4 states that the person using VSCode works on Wednesday and also Clue 5 indicates Carlos uses neither Emacs nor Sublime; this would suggest Ana might not use any of those possibilities (which leaves only JavaScript, Python, or Go since other editors weren't explicitly ruled out by clue 10 which does say Ana cannot work Tuesday. Given other clues about where the developers meet their languages -  Clue 4 says VSCode is used on Wednesday and also clue 11 indicates that Bruno can't use VSCode -  This leaves JS or Python working Friday leaving this option open for later.)
 
3. Question: What language does Diego use?
   Reasoning:
    - Ana works before Wednesday according to clue 7 which combined with clue 9 shows her is in the second day position (Thursday).
    Clue 11 states Bruno doesn’t do not work on VSCode. If we can assign JS or Python since they were given Friday for last, if we look at clue 2 about Ana being different days away than Tuesday then - which implies it was going to leave only one other language to be on Friday and the last two working days must give up their day to make this work. There are also constraints from other clues, so there is still ambiguity left, however - Diego must use Rust since Clues 3 & 12 gave restrictions that no Go developer uses vscode.
 
4. Question: On which day does Elisa work?
   Reasoning:
     Clue 6 and clue 1 show that the Java developers works on Friday at a time where Ana was placed Thursday by elimination of another language since Ana can’t do Tuesdays so only other two working days left out are Wednesday for VSCode or else Go - Since Elisa uses IntelliJ according to Clue 8. Then in Clues, this can be solved directly in conjunction with what we deduced before that - In clue 4 that works as Wednesday which fits perfectly since it leaves room for Ana to do Thursday (which matches up well considering all previous logic on elimination.)